In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
#곡률 계산
def compute_curvature(cnt, k=5):
    cnt = cnt[:, 0, :]
    curvatures = []
    for i in range(len(cnt)):
        p_prev = cnt[(i - k) % len(cnt)]
        p = cnt[i]
        p_next = cnt[(i + k) % len(cnt)]

        v1 = p_prev - p
        v2 = p_next - p

        v1_norm = np.linalg.norm(v1)
        v2_norm = np.linalg.norm(v2)
        if v1_norm == 0 or v2_norm == 0:
            curvatures.append(0)
            continue
        
        v1 = v1 / v1_norm
        v2 = v2 / v2_norm
        dot = np.clip(np.dot(v1, v2), -1.0, 1.0)
        angle = np.arccos(dot)

        curvatures.append(angle)

    return np.array(curvatures)

In [3]:
#모양 라벨링
def classify_shape(cnt, curvatures):
    x,y,w,h = cv2.boundingRect(cnt)
    aspect_ratio = w/h if w > h else h/w

    straight_ratio = np.sum(curvatures < 0.15) / len(curvatures)

    if aspect_ratio < 1.15:
        return "circle"
    if straight_ratio > 0.25:
        return "capsule"
    return "oval"

In [4]:
#알파 채널(투명도)로 윤곽을 찾은 후 모양 찾기
def contour_shape_from_alpha(img_path):
    rgba = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if rgba is None or rgba.shape[2] != 4:
        return []

    alpha = rgba[:, :, 3]
    _, mask = cv2.threshold(alpha, 10, 255, cv2.THRESH_BINARY)

    kernel = np.ones((5,5), np.uint8)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)

    cnts, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    shapes = []
    for cnt in cnts:
        if len(cnt) < 20:
            continue

        curv = compute_curvature(cnt)
        shape = classify_shape(cnt, curv)
        shapes.append(shape)

    return shapes

In [ ]:
# 폴더 전체 이미지 집계
def count_shapes_in_folder(folder_path):
    shape_count = {"circle": 0, "oval": 0, "capsule": 0}

    files = [f for f in os.listdir(folder_path) if f.lower().endswith((".png", ".jpg"))]

    for filename in files:
        img_path = os.path.join(folder_path, filename)

        shapes = contour_shape_from_alpha(img_path)

        for s in shapes:
            shape_count[s] += 1

    return shape_count